[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/crunch-numinous/blob/main/numinous/examples/quickstart.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/numinous/assets/banner.webp)

# Numinous — Binary Event Forecasting

Predict the probability that real-world events resolve **"Yes"**.

Events are sourced from prediction markets and cover politics, crypto, sports, weather, and more.

## Scoring

Predictions are evaluated with the **Brier score**:

$$\text{Brier} = (p - o)^2$$

where $p$ is your predicted probability and $o \in \{0, 1\}$ is the actual outcome. **Lower is better.**

| Score | Meaning |
|-------|---------|
| 0.00  | Perfect |
| 0.25  | Uninformative (always 0.5) |
| 1.00  | Worst possible |

Predictions are clipped to $[0.01, 0.99]$. Missing predictions are scored as 0.25.

## Setup

In [ ]:
!pip install crunch-numinous

## Gateway

Your notebook has **no direct internet access** in production. All external calls (LLMs, search, OSINT...) go through the **gateway**.

- **In production**: `SANDBOX_PROXY_URL` is set automatically — API costs are covered by Crunch
- **Locally**: you run the gateway yourself with your own API keys — most providers offer a free tier

See the [API Reference](https://github.com/crunchdao/numinous/blob/main/numinous/gateway/API_REFERENCE.md) for all available endpoints.

In [37]:
# --- LOCAL TESTING ONLY — remove this cell before submitting ---
# Option 1: set your API keys here
import os
# os.environ["OPENAI_API_KEY"] = ""           # https://platform.openai.com/api-keys
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."  # https://openrouter.ai/settings/keys
# os.environ["CHUTES_API_KEY"] = "..."            # https://chutes.ai/app

# Option 2: put them in a persistent env file (one KEY=VALUE per line)
# You can create it manually or run: crunch-numinous gateway configure
from pathlib import Path
print(f"Env file location: {Path.home() / '.crunch-numinous-gateway.env'}")

Env file location: /Users/aboutrig/.crunch-numinous-gateway.env


In [ ]:
!crunch-numinous gateway restart &

# To see HTTP requests and responses in the logs, use:
# !crunch-numinous gateway restart --debug &

# Then check the logs with:
# !crunch-numinous gateway logs --no-follow

## Your Model

Subclass `TrackerBase` and implement `_predict()`.

```
Input:  {"event_id": "...", "title": "Will X happen?", "yes_price": 0.65, ...}
Output: {"event_id": "...", "prediction": 0.72}
```

This example asks an LLM to estimate P(Yes) given the event details and current market price. It calls the gateway's OpenAI endpoint — swap the model or provider to experiment.

In [ ]:
import os
import re

import httpx
from numinous.tracker import TrackerBase


class LLMTracker(TrackerBase):
    """Asks an LLM to estimate P(Yes) for each event."""
    GATEWAY_URL = os.environ.get("SANDBOX_PROXY_URL", "http://localhost:8090")

    def _predict(self, subject):
        data = self._get_data(subject)
        if not isinstance(data, dict):
            return {"event_id": subject, "prediction": 0.5}

        event_id = data.get("event_id", subject)
        title = data.get("title", "")
        description = data.get("description", "")
        yes_price = float(data.get("yes_price", 0.5))

        prompt = (
            "You are a forecasting expert. Estimate the probability that "
            "this event resolves Yes.\n\n"
            f"Event: {title}\n"
            f"Description: {description}\n"
            f"Current market price: {yes_price}\n\n"
            "Reply with ONLY a decimal number between 0 and 1."
        )

        try:
            resp = httpx.post(
                f"{self.GATEWAY_URL}/api/gateway/openai/responses",
                json={
                    "model": "gpt-5-mini",
                    "input": [{"role": "user", "content": prompt}],
                    "max_output_tokens": 16,
                },
                timeout=180.0,
            )
            resp.raise_for_status()
            result = resp.json()

            # Extract text from OpenAI response
            text = ""
            for item in result.get("output", []):
                for content in (item.get("content") or []):
                    if content.get("text"):
                        text = content["text"]

            match = re.search(r"(0\.\d+|1\.0|0|1)", text)
            prediction = float(match.group(1)) if match else yes_price
        except Exception as e:
            print(f"  [{event_id}] LLM call failed: {e}, falling back to market price")
            prediction = yes_price

        return {"event_id": event_id, "prediction": max(0.0, min(1.0, prediction))}

## Test Locally
Feed sample events and check predictions.

In [43]:
SAMPLE_EVENTS = [
    {"event_id": "evt-btc-100k", "title": "Will BTC exceed $100k by end of March?",
     "yes_price": 0.65, "cutoff": "2026-04-01T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 150000.0, "metadata": {}},
    {"event_id": "evt-rain-nyc", "title": "Will it rain in NYC tomorrow?",
     "yes_price": 0.80, "cutoff": "2026-03-25T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 5000.0, "metadata": {}},
    {"event_id": "evt-fed-rate", "title": "Will the Fed cut rates in April?",
     "yes_price": 0.35, "cutoff": "2026-04-30T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 500000.0, "metadata": {}},
]

OUTCOMES = {"evt-btc-100k": 1, "evt-rain-nyc": 1, "evt-fed-rate": 0}

In [ ]:
from rich.table import Table
from rich.console import Console
from numinous.scoring import score_prediction, ForecastOutput, EventGroundTruth

model = LLMTracker()
scores = []

for event in SAMPLE_EVENTS:
    model.feed_update(event)
    result = model.predict(event["event_id"])
    score = score_prediction(
        ForecastOutput(event_id=event["event_id"], prediction=result["prediction"]),
        EventGroundTruth(event_id=event["event_id"], outcome=OUTCOMES[event["event_id"]]),
    )
    scores.append((event, score))

table = Table(title="Scoring Results")
for col in ("Event", "Market", "LLM", "Outcome", "Brier"):
    table.add_column(col, justify="right" if col != "Event" else "left")

for event, s in scores:
    table.add_row(event["title"][:50], f"{event['yes_price']:.2f}", f"{s.clipped_prediction:.3f}",
                  "Yes" if s.outcome else "No", f"{s.brier_score:.4f}")

avg = sum(s.brier_score for _, s in scores) / len(scores)
table.add_section()
table.add_row("[bold]Average Brier[/bold]", "", "", "", f"[bold]{avg:.4f}[/bold]")
table.add_row("[dim]Baseline (always 0.5)[/dim]", "", "", "", "[dim]0.2500[/dim]")

Console().print(table)

## For debugging

If you encounter any issues, you can view the gateway logs, which are useful for diagnosing API errors.

In [ ]:
!crunch-numinous gateway logs --no-follow

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Deploy it in real condition

### >> https://hub.crunchdao.com/competitions/numinous/submit/notebook

The platform will find your `TrackerBase` subclass automatically.

![Download and Submit Notebook](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook-deployment.gif)